In [1]:
import os
import json
from pathlib import Path

In [2]:
ENRON_DIR = "../../data/enronmail_raw"     # 主目录
OUTPUT_FILE = "mails.jsonl"

In [3]:
def parse_email(path):
    """
    将 enron 式邮件文本解析为 dict，包括 From, To, Subject, Date, Body
    """
    with open(path, "r", encoding="latin-1") as f:
        lines = f.readlines()

    headers = {}
    body_lines = []
    in_headers = True

    for line in lines:
        if in_headers:
            if line.strip() == "":
                in_headers = False
                continue
            # 解析 header 行
            if ":" in line:
                key, val = line.split(":", 1)
                headers[key.strip().lower()] = val.strip()
        else:
            body_lines.append(line.rstrip())

    return {
        "path": str(path),
        "from": headers.get("from", ""),
        "to": headers.get("to", ""),
        "subject": headers.get("subject", ""),
        "date": headers.get("date", ""),
        "ID": headers.get("message-id", ""),
        "body": "\n".join(body_lines),
        "rag_txt": "".join(lines[1:])
    }

def get_total_email(path):
    with open(path, "r", encoding="latin-1") as f:
        lines = f.readlines()

    return "".join(lines)

In [4]:
def all_emails(root_dir):
    """
    遍历所有文件夹并获得 email 文件路径
    """
    all_files = []
    for root, dirs, files in os.walk(root_dir):
        # print(f"Scanning {root} ...")
        for f in files:
            # Enron 数据基本都是 txt 或无后缀
            if f.lower().endswith(".txt") or f.lower().endswith(".") or "." not in f:
                all_files.append(Path(root) / f)
    return all_files

In [5]:
all_files_paths = all_emails(ENRON_DIR)
print(f"找到 {len(all_files_paths)} 封邮件样本，示例路径：{all_files_paths[:3]}")

找到 517401 封邮件样本，示例路径：[PosixPath('../../data/enronmail_raw/maildir/quigley-d/sent_items/7.'), PosixPath('../../data/enronmail_raw/maildir/quigley-d/sent_items/334.'), PosixPath('../../data/enronmail_raw/maildir/quigley-d/sent_items/79.')]


In [6]:
docs=[]

for email_path in all_files_paths:
    doc = parse_email(email_path)
    docs.append(doc)

In [7]:
docs[0]

{'path': '../../data/enronmail_raw/maildir/quigley-d/sent_items/7.',
 'from': 'dutch.quigley@enron.com',
 'to': 'markm@cajunusa.com',
 'subject': 'RE: Superbowl Party',
 'date': 'Wed, 30 Jan 2002 12:22:30 -0800 (PST)',
 'ID': '<10911795.1075841458665.JavaMail.evans@thyme>',
 'body': 'if you want to bring the veggie tray that would be great\notherwise just you and the family\n\nsee you there\n\ndq\n\n -----Original Message-----\nFrom: \t"Mark Molnar" <MarkM@cajunusa.com>@ENRON\nSent:\tWednesday, January 30, 2002 11:12 AM\nTo:\tQuigley, Dutch\nSubject:\tRE: Superbowl Party\n\nWhat should we bring?\n\n>>> "Quigley, Dutch" <Dutch.Quigley@ENRON.com> 01/29/02 03:35PM >>>\nthats really very cute\nand i liked your rhyme to boot\n\ni have tons of time\nwith enron in its decline\n\nplease wear your underwear\nwhile you cheer and no not swear\n\nwings would be great\nplease do not be late\n\non another note\nand this is no joke\n\nwhen is the dralion show\nso that i can plan to go\n\np ditty d\n\

In [8]:
rag_texts_output = [
    {
        "_id": item["ID"],
        "title": f"{item['subject']}",
        "text": item["rag_txt"],
        "metadata":{
            "path": item["path"],
            "from": item["from"],
            "to": item["to"],
            "date": item["date"]
        }
    }
    for item in docs]

In [9]:
rag_texts_output[0]

{'_id': '<10911795.1075841458665.JavaMail.evans@thyme>',
 'title': 'RE: Superbowl Party',
 'text': 'Date: Wed, 30 Jan 2002 12:22:30 -0800 (PST)\nFrom: dutch.quigley@enron.com\nTo: markm@cajunusa.com\nSubject: RE: Superbowl Party\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Quigley, Dutch </O=ENRON/OU=NA/CN=RECIPIENTS/CN=DQUIGLE>\nX-To: \'"Mark Molnar" <MarkM@cajunusa.com>@ENRON\'\nX-cc: \nX-bcc: \nX-Folder: \\ExMerge - Quigley, Dutch\\Sent Items\nX-Origin: QUIGLEY-D\nX-FileName: dutch quigley 6-26-02.PST\n\nif you want to bring the veggie tray that would be great\notherwise just you and the family\n\nsee you there\n\ndq\n\n -----Original Message-----\nFrom: \t"Mark Molnar" <MarkM@cajunusa.com>@ENRON  \nSent:\tWednesday, January 30, 2002 11:12 AM\nTo:\tQuigley, Dutch\nSubject:\tRE: Superbowl Party\n\nWhat should we bring?\n\n>>> "Quigley, Dutch" <Dutch.Quigley@ENRON.com> 01/29/02 03:35PM >>>\nthats really very cute\nand i liked

In [10]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for item in rag_texts_output:
        out.write(json.dumps(item, ensure_ascii=False) + "\n")